# Lab 13A — Delta Sharing: Provider y Consumer

**Sesion 13 | Databricks Data Engineer Associate**  
**Runtime minimo:** DBR 13.3 LTS (Unity Catalog habilitado)  
**Archivo fuente:** `ventas_externas.csv`

## Objetivos
- Cargar una tabla Gold desde un CSV como fuente de datos a compartir
- Crear un Share con filtro de particion por region
- Crear un Recipient y obtener su activation link
- Auditar el acceso con SHOW GRANTS y SHOW ALL IN SHARE
- **[NUEVO]** Consumir el Share desde otro workspace Databricks (opcion A: Databricks-to-Databricks)
- **[NUEVO]** Consumir el Share con el cliente open-source delta-sharing (opcion B: cualquier plataforma)

## Setup previo
Subir el archivo al Volume:  
`/Volumes/dbassociate/default/vol_landing/sesion13/ventas_externas.csv`

---
## Arquitectura del lab

```
WORKSPACE PROVEEDOR (este notebook)          WORKSPACE RECEPTOR
--------------------------------------------  ----------------------------
UC Metastore A                               UC Metastore B (otro tenant)
  dbassociate.gold.ventas_externas
        |                                           ^
  CREATE SHARE share_ventas_latam                   |
  ALTER SHARE ADD TABLE ... PARTITION(MX)           |  CREATE PROVIDER
  CREATE RECIPIENT cliente_externo_mx    --link-->  |  CREATE FOREIGN CATALOG
  GRANT SELECT ON SHARE TO RECIPIENT                |  SELECT * FROM catalogo.*
```

## Paso 0 — Verificacion del entorno

In [0]:
display(dbutils.fs.ls("/Volumes/dbassociate/default/vol_landing/sesion13"))

## Paso 1 — Constantes del lab

In [0]:
CATALOG     = "dbassociate"
SCHEMA_GOLD = "gold"
VOLUME_PATH = "/Volumes/dbassociate/default/vol_landing/sesion13"

print(f"Volume path   : {VOLUME_PATH}")
print(f"Tabla destino : {CATALOG}.{SCHEMA_GOLD}.ventas_externas")

## Paso 2 — Cargar CSV con esquema explicito

Antipatron a evitar: `inferSchema=True` en archivos grandes introduce un scan completo antes de la lectura real.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema_ventas = StructType([
    StructField("id_venta",  StringType(),  False),
    StructField("fecha",     StringType(),  True),
    StructField("region",    StringType(),  True),
    StructField("producto",  StringType(),  True),
    StructField("cantidad",  IntegerType(), True),
    StructField("monto",     DoubleType(),  True),
    StructField("canal",     StringType(),  True),
])

df_ventas = (
    spark.read
    .option("header", True)
    .schema(schema_ventas)
    .csv(f"{VOLUME_PATH}/ventas_externas.csv")
)

print(f"Filas cargadas    : {df_ventas.count()}")
print(f"Regiones presentes: {sorted([r.region for r in df_ventas.select('region').distinct().collect()])}")
display(df_ventas.limit(5))

## Paso 3 — Persistir como tabla Gold en Unity Catalog

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA_GOLD}.ventas_externas")

(
    df_ventas
    .write.format("delta")
    .mode("overwrite")
    .partitionBy("region")
    .saveAsTable(f"{CATALOG}.{SCHEMA_GOLD}.ventas_externas")
)

print(f"Tabla creada: {CATALOG}.{SCHEMA_GOLD}.ventas_externas")
spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCHEMA_GOLD}.ventas_externas").show()

In [0]:
%sql
select * from dbassociate.gold.ventas_externas;

## Paso 4 — Crear el Share

Un Share es un contenedor logico en el metastore. No copia datos — apunta a las tablas originales.

In [0]:
spark.sql("CREATE SHARE IF NOT EXISTS share_ventas_latam COMMENT 'Ventas por region para partners externos'")

print("Share creado. Shares en el metastore:")
spark.sql("SHOW SHARES").show(truncate=False)

## Paso 5 — Agregar tabla al Share con partition filter

El `PARTITION (region = 'MX')` es el mecanismo de Row-Level Security del Share:  
el recipient no puede ver datos de CO, PE ni AR aunque pertenezcan a la misma tabla.

In [0]:
spark.sql(f"""
    ALTER SHARE share_ventas_latam
    ADD TABLE {CATALOG}.{SCHEMA_GOLD}.ventas_externas
    PARTITION (region = 'MX')
""")

print("Tabla agregada al Share con filtro de particion region='MX'")
print("")
spark.sql("SHOW ALL IN SHARE share_ventas_latam").show(truncate=False)

## Paso 6 — Crear Recipient y obtener activation link

El activation link debe enviarse al receptor por un canal seguro.  
Una vez descargado el profile.json el link se invalida automaticamente.

In [0]:
%sql
SELECT CURRENT_METASTORE();

In [0]:
# =============================================================
# SHARING DATABRICKS-TO-DATABRICKS
# =============================================================
# Sharing identifier del workspace receptor (obtenido con SELECT CURRENT_METASTORE())
# Formato: <cloud>:<region>:<metastore-uuid>
# =============================================================

SHARING_ID_RECEPTOR = "azure:brazilsouth:11b23b7c-3196-44fe-998f-b6eb554d57d7"

# Eliminar recipient anterior si existe
spark.sql("DROP RECIPIENT IF EXISTS cliente_externo_mx")

spark.sql(f"""
    CREATE RECIPIENT IF NOT EXISTS cliente_externo_mx
    USING ID '{SHARING_ID_RECEPTOR}'
    COMMENT 'Partner externo con acceso a ventas de region MX'
""")

print(f"Recipient D2D creado con sharing identifier: {SHARING_ID_RECEPTOR}")
print("authentication_type: DATABRICKS (no requiere activation link)")
print("")
result = spark.sql("DESCRIBE RECIPIENT cliente_externo_mx")
display(result)

## Paso 7 — Otorgar SELECT al Recipient sobre el Share

Sin este GRANT el recipient tiene un token valido pero no puede leer ninguna tabla del Share.

In [0]:
spark.sql("GRANT SELECT ON SHARE share_ventas_latam TO RECIPIENT cliente_externo_mx")

print("Permiso otorgado.")
print("\nPermisos actuales sobre el Share:")
spark.sql("SHOW GRANTS ON SHARE share_ventas_latam").show(truncate=False)

## Paso 8 — Auditoria del Share

In [0]:
print("=== RECIPIENTS EN EL METASTORE ===")
spark.sql("SHOW RECIPIENTS").show(truncate=False)

print("\n=== CONTENIDO DEL SHARE ===")
spark.sql("SHOW ALL IN SHARE share_ventas_latam").show(truncate=False)

print("\n=== GRANTS SOBRE EL SHARE ===")
spark.sql("SHOW GRANTS ON SHARE share_ventas_latam").show(truncate=False)

---
# PARTE B — CONSUMER: Consumir el Share desde otro workspace

A partir de aqui los comandos representan lo que ejecuta el **administrador del workspace receptor**.  
Existen dos modalidades segun el tipo de receptor:

| Opcion | Receptor | Mecanismo |
|--------|----------|-----------|
| A | Databricks con Unity Catalog | `CREATE PROVIDER` + `CREATE FOREIGN CATALOG` (SQL en UC) |
| B | Cualquier plataforma (Python, Pandas, Spark OSS) | cliente `delta-sharing` + `profile.json` |

## Paso 10 — Opcion A: Crear DELTASHARING_CATALOG con el PROVIDER

**Donde ejecutar:** workspace receptor (metastore B).

In [0]:
print("Ver shares disponibles del provider 'dmc'")

df_shares = spark.sql("SHOW SHARES IN PROVIDER dmc")
df_shares.show(truncate=False)

In [0]:
print("Crear catálogo desde el share 'share_ventas_latam'")

# Crear el catálogo usando el share del provider dmc
# El nombre del catálogo puedes cambiarlo según tu convención
catalog_name = "cat_dmc_ventas_latam"

try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name} USING SHARE dmc.share_ventas_latam")
    print(f"\n Catálogo '{catalog_name}' creado exitosamente desde el share 'dmc.share_ventas_latam'")
    
    # Mostrar schemas disponibles
    print(f"\nSchemas en el catálogo '{catalog_name}':")
    spark.sql(f"SHOW SCHEMAS IN {catalog_name}").show(truncate=False)
    
    # Mostrar tablas en cada schema
    schemas = [row.databaseName for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
    for schema in schemas:
        if schema not in ('information_schema', 'default'):
            print(f"\nTablas en {catalog_name}.{schema}:")
            spark.sql(f"SHOW TABLES IN {catalog_name}.{schema}").show(truncate=False)
except Exception as e:
    print(f"\n Error al crear el catálogo: {e}")
    print("\n Posibles causas:")
    print("  - No tienes permisos CREATE CATALOG en el metastore")
    print("  - El nombre del catálogo ya existe")
    print("  - El share no está accesible")

## Paso 13 — Limpieza

In [0]:
%sql
DROP TABLE IF EXISTS dbassociate.gold.ventas_externas;

In [0]:
spark.sql("REVOKE SELECT ON SHARE share_ventas_latam FROM RECIPIENT cliente_externo_mx")
spark.sql("DROP RECIPIENT IF EXISTS cliente_externo_mx")
spark.sql("DROP SHARE IF EXISTS share_ventas_latam")
spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{SCHEMA_GOLD}.ventas_externas")

import os
if os.path.exists("/tmp/delta_sharing_profile.json"):
    os.remove("/tmp/delta_sharing_profile.json")

print("Recursos del lab eliminados:")
print("  - RECIPIENT cliente_externo_mx")
print("  - SHARE share_ventas_latam")
print(f"  - TABLE {CATALOG}.{SCHEMA_GOLD}.ventas_externas")
print("  - /tmp/delta_sharing_profile.json")

## Puntos clave del examen

**Lado proveedor:**
1. `CREATE SHARE` crea el contenedor logico — no copia datos, solo define que tablas son visibles
2. `PARTITION (col = 'valor')` en `ALTER SHARE` restringe las filas visibles por recipient
3. `CREATE RECIPIENT` genera el activation link — quien tenga el `profile.json` tiene acceso
4. `GRANT SELECT ON SHARE` es obligatorio — sin el GRANT el recipient no puede leer aunque tenga el token
5. `ROTATE ACTIVATION LINK` invalida el link anterior de forma inmediata — usar ante incidentes de seguridad
6. Tablas de `hive_metastore` NO pueden incluirse en un Share — solo tablas de Unity Catalog

**Lado consumer:**
7. `CREATE PROVIDER` + `CREATE FOREIGN CATALOG` es el flujo Databricks-to-Databricks — el receptor consulta el Share como si fuera un catalogo local de solo lectura
8. El Foreign Catalog es **inmutable** desde el receptor — INSERT, UPDATE y DELETE no estan permitidos
9. El partition filter del Share se aplica en el proveedor; el receptor **no puede ver ni saltarse** esa restriccion
10. El cliente open-source `delta-sharing` permite consumir Shares desde Python, Pandas o Spark OSS sin necesidad de Unity Catalog en el receptor
11. El `profile.json` contiene el endpoint y el bearerToken — debe tratarse como una credencial y no exponerse en notebooks ni logs